## Plot Circumpolar SWOT Mosiacs like figure 4 from Fraser and Wongpan Fast Ice Review

In [ ]:
from pathlib import Path
import rioxarray as rxr
import matplotlib.pyplot as plt

# Root directory containing all longitude-sector folders
data_dir = Path("/g/data/gv90/ae8456/SWOT_SEAICE/DATA")

sector_dirs = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name[0].isdigit()
]

for d in sector_dirs:
    print(d.name)

print(len(sector_dirs))

In [ ]:
tif_files = []

for sector_dir in sector_dirs:
    tif_files.extend(
        sector_dir.glob(
            "SWOT_FREEBOARD_mosaic_cycle_new*/*_freeboard_mosaic.tif"
        )
    )

tif_files = sorted(tif_files)

print(f"Found {len(tif_files)} TIFF files")

## Should be 38 if you split the peninsula 

In [ ]:
for sector_dir in sector_dirs:
    matches = list(
        sector_dir.glob(
            "SWOT_FREEBOARD_mosaic_cycle_new*/*_freeboard_mosaic.tif"
        )
    )

    if len(matches) == 0:
        print(f"No TIFF found: {sector_dir.name}")

## if it is just 60-50W and 70-60W thats fine. we had to spilt the peninsula so each of these have seperate directories for the splits

In [ ]:
freeboard = {}

for tif in tif_files:
    sector = tif.relative_to(data_dir).parts[0]

    da = rxr.open_rasterio(
        tif,
        masked=True
    )

    # Remove singleton band dimension
    if "band" in da.dims and da.sizes["band"] == 1:
        da = da.squeeze("band", drop=True)

    freeboard[sector] = da

    print(
        f"{sector:15s} "
        f"shape={da.shape} "
        f"CRS={da.rio.crs}"
    )

In [ ]:
tif_files

In [ ]:
xmin_all = []
xmax_all = []
ymin_all = []
ymax_all = []

for tif in tif_files:

    with rxr.open_rasterio(tif, masked=True) as da:
        xmin, ymin, xmax, ymax = da.rio.bounds()

        xmin_all.append(xmin)
        xmax_all.append(xmax)
        ymin_all.append(ymin)
        ymax_all.append(ymax)

xmin = min(xmin_all)
xmax = max(xmax_all)
ymin = min(ymin_all)
ymax = max(ymax_all)

print(f"x limits: {xmin:.0f}, {xmax:.0f}")
print(f"y limits: {ymin:.0f}, {ymax:.0f}")

In [ ]:
def plot_swot(ax, tif_files, data_dir, step=1):

    for tif in tif_files:

        sector = tif.relative_to(data_dir).parts[0]

        with rxr.open_rasterio(tif, masked=True) as da:

            da = da.squeeze("band", drop=True)

            da_small = da.isel(
                x=slice(None, None, step),
                y=slice(None, None, step)
            )

            x = da_small.x.values
            y = da_small.y.values

            ax.imshow(
                da_small.values,
                extent=[
                    x.min(),
                    x.max(),
                    y.min(),
                    y.max()
                ],
                origin="upper",
                vmin=0,
                vmax=0.6,
                cmap="viridis"
            )

    ax.set_aspect("equal")

In [ ]:
fig = plt.figure(figsize=(14, 14))

# --------------------------------------------------
# CENTRAL ANTARCTIC MAP
# --------------------------------------------------

ax_main = fig.add_axes([0.25, 0.25, 0.50, 0.50])

ax_main.set_xlim(xmin, xmax)
ax_main.set_ylim(ymin, ymax)

plot_swot(
    ax_main,
    tif_files,
    data_dir,
    step=10
)

ax_main.set_xticks([])
ax_main.set_yticks([])


# --------------------------------------------------
# INSET POSITIONS
# --------------------------------------------------

inset_positions = {
    "a": [0.02, 0.76, 0.22, 0.18],
    "b": [0.26, 0.80, 0.22, 0.18],
    "c": [0.52, 0.80, 0.22, 0.18],
    "d": [0.76, 0.76, 0.22, 0.18],

    "e": [0.78, 0.52, 0.20, 0.18],
    "f": [0.78, 0.28, 0.20, 0.18],

    "g": [0.76, 0.04, 0.22, 0.18],
    "h": [0.52, 0.02, 0.22, 0.18],
    "i": [0.26, 0.02, 0.22, 0.18],
    "j": [0.02, 0.04, 0.22, 0.18],

    "k": [0.00, 0.28, 0.22, 0.18],
    "l": [0.00, 0.52, 0.22, 0.18],
}


# --------------------------------------------------
# INSET EXTENTS
# --------------------------------------------------
956223,2323116
inset_extents = {
    "a": [-841332, -602980, 1086729, 1423340],
    "b": [333889, 511924, 2099482, 2212334],
    "c": [578322, 683148, 2100455, 2208199],
    "d": [1159325, 1272177, 1920089, 2017619],
    "e": [1314740, 1498126, 1698276, 1977002],
    "f": [1631895, 1800200, 1674198, 1848827],
    "g": [2033202, 2323116, 956223, 1517566],
    #"h": [333889, 511924, 2212334, 2099482],
    #"i": [333889, 511924, 2212334, 2099482],
    #"j": [333889, 511924, 2212334, 2099482],
    #"k": [333889, 511924, 2212334, 2099482],
    #"l": [333889, 511924, 2212334, 2099482],
}


# --------------------------------------------------
# CREATE INSETS
# --------------------------------------------------

inset_axes = {}

for label, extent in inset_extents.items():

    pos = inset_positions[label]

    ax = fig.add_axes(pos)

    xi0, xi1, yi0, yi1 = extent

    ax.set_xlim(xi0, xi1)
    ax.set_ylim(yi0, yi1)

    plot_swot(
        ax,
        tif_files,
        data_dir,
        step=1
    )

    ax.set_xticks([])
    ax.set_yticks([])

    ax.text(
        0.02,
        0.96,
        f"({label})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        fontweight="bold"
    )

    inset_axes[label] = ax

output_path = "/g/data/jk72/ae8456/Figures/SWOT/Cycle21_mosiac_intermediate.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0,
    transparent=False
    )
plt.show()